<a href="https://colab.research.google.com/github/Yukselendincer/datasceinceproject/blob/main/DataLakeYellowTaxi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Gerekli Java ortamını ve paketleri sessizce kur
!apt-get update -qq > /dev/null
!apt-get install openjdk-17-jdk-headless -qq > /dev/null
!pip install -q pyspark==3.5.1 delta-spark==3.2.0 duckdb

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# 3 milyon satır için 6GB sürücü hafızası ayırıyoruz
builder = SparkSession.builder \
    .appName("NYCTaxi_Medallion") \
    .master("local[*]") \
    .config("spark.driver.memory", "6g") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# Medallion klasör hiyerarşisi
base_dir = "/content/nyc_datalake"
for layer in ["bronze", "silver", "gold"]:
    os.makedirs(f"{base_dir}/{layer}", exist_ok=True)

print("✓ SparkSession hazır, dizinler oluşturuldu.")

In [ ]:
%%bash
# Resmi TLC CloudFront adresinden ham veriyi indir
curl -sSL "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet" \
     -o /content/nyc_datalake/bronze/yellow_tripdata_2024_01_raw.parquet

ls -lh /content/nyc_datalake/bronze/

In [ ]:
from pyspark.sql.functions import col, to_date, datediff

# Bronze verisini oku
df_bronze = spark.read.parquet(f"{base_dir}/bronze/yellow_tripdata_2024_01_raw.parquet")
initial_count = df_bronze.count()

# Veri Kalitesi Kuralları (Silver):
# 1. Yolcu sayısı > 0
# 2. Mesafe > 0 mil
# 3. Toplam ücret > 0 $
# 4. Sadece Ocak 2024 seyahatleri (bazı sensör hataları 2008 veya 2088 gösterebilir)
df_silver = df_bronze.filter(
    (col("passenger_count") > 0) &
    (col("trip_distance") > 0) &
    (col("total_amount") > 0) &
    (col("tpep_pickup_datetime") >= "2024-01-01") &
    (col("tpep_pickup_datetime") < "2024-02-01")
).withColumn("pickup_date", to_date(col("tpep_pickup_datetime")))

# Silver katmanına Delta formatında yaz (Tarihe göre partition uygulayarak)
silver_path = f"{base_dir}/silver/yellow_trips"
df_silver.write \
    .format("delta") \
    .partitionBy("pickup_date") \
    .mode("overwrite") \
    .save(silver_path)

cleaned_count = spark.read.format("delta").load(silver_path).count()

print(f"Ham Kayıt Sayısı      : {initial_count:,}")
print(f"Temizlenmiş (Silver)  : {cleaned_count:,}")
print(f"Elenen Hatalı Kayıt   : {initial_count - cleaned_count:,}")

In [ ]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

df_silver_loaded = spark.read.format("delta").load(silver_path)

df_gold_daily = df_silver_loaded.groupBy("pickup_date").agg(
    count("*").alias("total_trips"),
    _round(avg("trip_distance"), 2).alias("avg_distance_miles"),
    _round(_sum("total_amount"), 2).alias("total_revenue_usd"),
    _round(avg("tip_amount"), 2).alias("avg_tip_usd")
).orderBy("pickup_date")

gold_path = f"{base_dir}/gold/daily_taxi_metrics"
df_gold_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

print("✓ Gold metrikleri Delta formatında kaydedildi.")

In [ ]:
import duckdb

con = duckdb.connect()
con.install_extension("delta")
con.load_extension("delta")

query = f"""
    SELECT
        pickup_date,
        total_trips,
        avg_distance_miles,
        total_revenue_usd,
        avg_tip_usd
    FROM delta_scan('{gold_path}')
    ORDER BY pickup_date ASC
    LIMIT 10
"""

df_report = con.execute(query).df()
display(df_report)

In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df_report)